In [61]:
import os
import pandas as pd
import re
from collections import defaultdict
from pathlib import Path

def get_indiv_concepts(formula) -> list:
    concepts = []
    concps = re.findall(r'(?<!\bNOT\s)(?:\b(?:hyp|pre|oth):[^\s)]+)', formula)
    for c in concps:
        try:
            end_idx = c.index(')')
        except:
            end_idx = len(c)
        concepts.append(c[:end_idx])
    return concepts

def load_csv_data(filepath):
    """Load CSV and extract unit-concept mappings."""
    df = pd.read_csv(filepath)
    unit_concepts = defaultdict(set)
    
    for _, row in df.iterrows():
        unit = row['unit']
        formula = row['best_name']
        concepts = get_indiv_concepts(formula)
        unit_concepts[unit].update(concepts)
    
    return unit_concepts

def analyze_concept_loss(base_dir, model, exp, method, clusters=['Cluster1', 'Cluster2', 'Cluster3'],filename=None):
    """
    Analyze which concepts are lost between different pruning percentages.
    
    Args:
        base_dir: Base directory path
        model: Model name
        exp: Experiment name
        method: Method name (e.g., 'method1', 'method2')
        clusters: List of cluster names
    
    Returns:
        Dictionary containing loss analysis for each cluster
    """
    # Define pruning percentages
    pruning_percentages = ['0.0%Pruned', '25.0%Pruned', '43.75%Pruned', '57.812%Pruned', '68.359%Pruned', '76.27%Pruned']
    
    results = {}
    
    for cluster in clusters:
        print(f"\n{'='*60}")
        print(f"Analyzing {cluster} for {model}/{exp}/{method}")
        print(f"{'='*60}")
        
        cluster_results = {
            'concepts_by_pruning': {},
            'loss_from_baseline': {},
            'sequential_loss': {},
            'cumulative_loss': {}
        }
        
        # Load concepts for each pruning percentage
        baseline_concepts = None
        prev_concepts = None
        
        for pruning_pct in pruning_percentages:
            filepath = os.path.join(base_dir, model, exp, method,filename, 'Expls', pruning_pct, f'{cluster}IOUS1024N.csv')
            
            if not os.path.exists(filepath):
                print(f"Warning: File not found - {filepath}")
                continue
            
            # Load unit-concept mappings
            unit_concepts = load_csv_data(filepath)
            
            # Get all unique concepts at this pruning level
            all_concepts = set()
            for concepts in unit_concepts.values():
                all_concepts.update(concepts)
            
            cluster_results['concepts_by_pruning'][pruning_pct] = {
                'all_concepts': all_concepts,
                'num_concepts': len(all_concepts),
                'unit_concepts': unit_concepts
            }
            
            # Set baseline (0.0% pruned)
            if baseline_concepts is None:
                baseline_concepts = all_concepts
                print(f"\n{pruning_pct}: {len(all_concepts)} concepts (BASELINE)")
            else:
                # Calculate loss from baseline
                lost_from_baseline = baseline_concepts - all_concepts
                retained_from_baseline = baseline_concepts & all_concepts
                
                cluster_results['loss_from_baseline'][pruning_pct] = {
                    'lost_concepts': lost_from_baseline,
                    'num_lost': len(lost_from_baseline),
                    'retained_concepts': retained_from_baseline,
                    'num_retained': len(retained_from_baseline),
                    'loss_percentage': (len(lost_from_baseline) / len(baseline_concepts) * 100) if baseline_concepts else 0
                }
                
                print(f"\n{pruning_pct}: {len(all_concepts)} concepts")
                print(f"  Lost from baseline: {len(lost_from_baseline)} ({cluster_results['loss_from_baseline'][pruning_pct]['loss_percentage']:.2f}%)")
                print(f"  Retained from baseline: {len(retained_from_baseline)}")
                
                # Calculate sequential loss (compared to previous pruning level)
                if prev_concepts is not None:
                    lost_sequential = prev_concepts - all_concepts
                    
                    cluster_results['sequential_loss'][pruning_pct] = {
                        'lost_concepts': lost_sequential,
                        'num_lost': len(lost_sequential),
                        'loss_percentage': (len(lost_sequential) / len(prev_concepts) * 100) if prev_concepts else 0
                    }
                    
                    print(f"  Lost since previous: {len(lost_sequential)} ({cluster_results['sequential_loss'][pruning_pct]['loss_percentage']:.2f}%)")
            
            prev_concepts = all_concepts
        
        results[cluster] = cluster_results
    
    return results
from collections import defaultdict
def generate_loss_summary(loss_results, save_path=None):
    """
    Generate a summary table of concept loss across pruning percentages.
    
    Args:
        loss_results: Results from analyze_concept_loss
        save_path: Optional path to save CSV summary
    """
    summary_data = []
    raw_concepts = defaultdict(lambda: defaultdict(list))
    max_s=defaultdict(int)
    for cluster, cluster_data in loss_results.items():
        for pct in cluster_data['loss_from_baseline'].keys():
            raw_concepts[cluster][pct]= list(cluster_data['loss_from_baseline'][pct]['lost_concepts'])
            max_s[cluster] = max(max_s[cluster], len(raw_concepts[cluster][pct]))
        for pruning_pct in cluster_data['concepts_by_pruning'].keys():
            row = {
                'Cluster': cluster,
                'Pruning_Percentage': pruning_pct,
                'Total_Concepts': cluster_data['concepts_by_pruning'][pruning_pct]['num_concepts']
            }
            
            if pruning_pct in cluster_data['loss_from_baseline']:
                row['Lost_from_Baseline'] = cluster_data['loss_from_baseline'][pruning_pct]['num_lost']
                row['Lost_from_Baseline_Pct'] = cluster_data['loss_from_baseline'][pruning_pct]['loss_percentage']
                row['Retained_from_Baseline'] = cluster_data['loss_from_baseline'][pruning_pct]['num_retained']
            else:
                row['Lost_from_Baseline'] = 0
                row['Lost_from_Baseline_Pct'] = 0
                row['Retained_from_Baseline'] = row['Total_Concepts']
            
            if pruning_pct in cluster_data['sequential_loss']:
                row['Lost_Sequential'] = cluster_data['sequential_loss'][pruning_pct]['num_lost']
                row['Lost_Sequential_Pct'] = cluster_data['sequential_loss'][pruning_pct]['loss_percentage']
            else:
                row['Lost_Sequential'] = 0
                row['Lost_Sequential_Pct'] = 0
            
            summary_data.append(row)
    
    for cluster in raw_concepts:
        for pct in raw_concepts[cluster]:
            dif = max_s[cluster] - len(raw_concepts[cluster][pct]) 
            if dif > 0:
                for i in range(dif):
                    raw_concepts[cluster][pct].append('')
                
        pd.DataFrame({k: sorted(v) for k, v in raw_concepts[cluster].items()}).to_csv(f"{save_path}_Cluster{cluster}_concepts_lost.csv")
    summary_df = pd.DataFrame(summary_data)
    pd.DataFrame(raw_concepts).to_csv("Raw_concepts_lost_to_pruning.csv")
    if save_path:
        summary_df.to_csv(save_path, index=False)
        print(f"\nSummary saved to {save_path}_concept_loss_summary.csv")
    
    return summary_df

def get_lost_concepts_details(loss_results, cluster, pruning_pct):
    """
    Get detailed list of lost concepts for a specific cluster and pruning percentage.
    
    Args:
        loss_results: Results from analyze_concept_loss
        cluster: Cluster name
        pruning_pct: Pruning percentage (e.g., '25.0%Pruned')
    
    Returns:
        Set of lost concepts
    """
    if cluster not in loss_results:
        print(f"Cluster {cluster} not found in results")
        return set()
    
    if pruning_pct not in loss_results[cluster]['loss_from_baseline']:
        print(f"Pruning percentage {pruning_pct} not found for {cluster}")
        return set()
    
    return loss_results[cluster]['loss_from_baseline'][pruning_pct]['lost_concepts']

# Example usage:
if __name__ == "__main__":
    # Set your paths
    base_dir = "/workspace/CCE_NLI"
    model = "LLAMA"
    exp = "exp"
    method = "lottery_ticket"
    filename='Run0.25_2'
    
    # Analyze concept loss
    loss_results = analyze_concept_loss(base_dir, model, exp, method, filename=filename)
    
    # Generate summary table
    summary_df = generate_loss_summary(loss_results, save_path=f'{model}_{method}_{filename}')
    print("\nSummary Table:")
    print(summary_df)

    # Get specific lost concepts
    lost_concepts_25 = get_lost_concepts_details(loss_results, 'Cluster1', '25.0%Pruned')
    print(f"\nConcepts lost at 25% pruning in cluster1: {len(lost_concepts_25)}")
    print(f"Examples: {list(lost_concepts_25)[:10]}")


Analyzing Cluster1 for LLAMA/exp/lottery_ticket

0.0%Pruned: 194 concepts (BASELINE)

25.0%Pruned: 135 concepts
  Lost from baseline: 80 (41.24%)
  Retained from baseline: 114
  Lost since previous: 80 (41.24%)

43.75%Pruned: 129 concepts
  Lost from baseline: 84 (43.30%)
  Retained from baseline: 110
  Lost since previous: 35 (25.93%)

57.812%Pruned: 158 concepts
  Lost from baseline: 77 (39.69%)
  Retained from baseline: 117
  Lost since previous: 28 (21.71%)

68.359%Pruned: 180 concepts
  Lost from baseline: 57 (29.38%)
  Retained from baseline: 137
  Lost since previous: 42 (26.58%)

76.27%Pruned: 170 concepts
  Lost from baseline: 65 (33.51%)
  Retained from baseline: 129
  Lost since previous: 56 (31.11%)

Analyzing Cluster2 for LLAMA/exp/lottery_ticket

0.0%Pruned: 233 concepts (BASELINE)

25.0%Pruned: 206 concepts
  Lost from baseline: 79 (33.91%)
  Retained from baseline: 154
  Lost since previous: 79 (33.91%)

43.75%Pruned: 221 concepts
  Lost from baseline: 76 (32.62%)
  Re

In [71]:
c1_lost_wanda = pd.read_csv("LLAMA_wanda_Run0.25_2_ClusterCluster1_concepts_lost.csv")
c1_lost_lth = pd.read_csv("LLAMA_lottery_ticket_Run0.25_2_ClusterCluster1_concepts_lost.csv")
lost=set(c1_lost_lth['25.0%Pruned'])
for i in c1_lost_lth:
    if '%' not in i: continue
    lost = lost.intersection(set(c1_lost_lth[i]))
print(f"Num lost in lth {len(lost)}")
for i in c1_lost_wanda:
    if '%' not in i: continue
    lost = lost.intersection(set(c1_lost_wanda[i]))
print(f"Num lost in wanda and lth: {len(lost)}")
print(lost)

Num lost in lth 32
Num lost in wanda and lth: 13
{'hyp:tok:green', 'pre:tok:top', 'pre:tok:there', 'pre:tok:lake', 'pre:tok:players', 'pre:tok:tree', 'hyp:tok:around', 'pre:tok:wall', 'pre:tok:performing', 'hyp:tok:driving', 'hyp:tok:female', 'hyp:tok:cats', 'pre:tok:out'}


In [67]:
len(preserved) #% of concepts that are lost once you rpune (lost at all iters)

33

In [72]:
c2_lost_wanda = pd.read_csv("LLAMA_wanda_Run0.25_2_ClusterCluster2_concepts_lost.csv")
c2_lost_lth= pd.read_csv("LLAMA_lottery_ticket_Run0.25_2_ClusterCluster2_concepts_lost.csv")
lost=set(c2_lost_lth['25.0%Pruned'])
for i in c2_lost_lth:
    if '%' not in i: continue
    lost = lost.intersection(set(c2_lost_lth[i]))
print(f"Num lost in lth {len(lost)}")
for i in c2_lost_wanda:
    if '%' not in i: continue
    lost = lost.intersection(set(c2_lost_wanda[i]))
print(f"Num lost in wanda and lth: {len(lost)}")
print(lost)

Num lost in lth 21
Num lost in wanda and lth: 8
{'pre:tok:church', 'pre:tok:vendor', 'pre:tok:workers', 'hyp:tok:air', 'hyp:tok:guitar', 'pre:tok:jacket', 'hyp:tok:couple', 'pre:tag:ex'}


In [73]:
c3_lost_wanda = pd.read_csv("LLAMA_wanda_Run0.25_2_ClusterCluster3_concepts_lost.csv")
c3_lost_lth = pd.read_csv("LLAMA_lottery_ticket_Run0.25_2_ClusterCluster3_concepts_lost.csv")
lost=set(c3_lost_lth['25.0%Pruned'])
for i in c3_lost_lth:
    if '%' not in i: continue
    lost = lost.intersection(set(c3_lost_lth[i]))
print(f"Num lost in lth {len(lost)}")
for i in c3_lost_wanda:
    if '%' not in i: continue
    lost = lost.intersection(set(c3_lost_wanda[i]))
print(f"Num lost in wanda and lth: {len(lost)}")
print(lost)

Num lost in lth 27
Num lost in wanda and lth: 10
{'hyp:tok:construction', 'pre:tok:graffiti', 'hyp:tok:summer', 'pre:tok:skier', 'pre:tok:board', 'pre:tok:surfing', 'pre:tok:shoulders', 'pre:tok:martial', 'pre:tok:flying', 'hyp:tok:runs'}


In [123]:
import os
import pandas as pd
import re
from collections import defaultdict
from pathlib import Path

def get_indiv_concepts(formula) -> list:
    concepts = []
    concps = re.findall(r'(?<!\bNOT\s)(?:\b(?:hyp|pre|oth):[^\s)]+)', formula)
    for c in concps:
        try:
            end_idx = c.index(')')
        except:
            end_idx = len(c)
        concepts.append(c[:end_idx])
    return concepts

def load_csv_data(filepath):
    """Load CSV and extract unit-concept mappings."""
    df = pd.read_csv(filepath)
    unit_concepts = defaultdict(set)
    
    for _, row in df.iterrows():
        unit = row['unit']
        formula = row['best_name']
        concepts = get_indiv_concepts(formula)
        unit_concepts[unit].update(concepts)
    
    return unit_concepts

def get_all_concepts_across_clusters(base_dir, model, exp, method, pruning_pct, 
                                     clusters=['cluster1', 'cluster2', 'cluster3'],filename=None):
    """
    Get all unique concepts across all clusters for a given pruning percentage.
    
    Args:
        base_dir: Base directory path
        model: Model name
        exp: Experiment name
        method: Method name
        pruning_pct: Pruning percentage (e.g., '0.0%Pruned')
        clusters: List of cluster names
    
    Returns:
        Set of all unique concepts across all clusters
    """
    all_concepts = set()
    
    for cluster in clusters:
        filepath = os.path.join(base_dir, model, exp, method,filename,   'Expls',pruning_pct, f'{cluster}IOUS1024N.csv')
        
        if not os.path.exists(filepath):
            print(f"Warning: File not found - {filepath}")
            continue
        
        # Load unit-concept mappings
        unit_concepts = load_csv_data(filepath)
        
        # Get all concepts in this cluster
        for concepts in unit_concepts.values():
            all_concepts.update(concepts)
    
    return all_concepts

def analyze_concept_loss_across_clusters(base_dir, model, exp, method, clusters=['Cluster1', 'Cluster2', 'Cluster3'],filename=None):
    """
    Analyze which concepts are lost across ALL clusters between different pruning percentages.
    A concept is "lost" if it appears in ANY cluster at 0.0%Pruned but doesn't appear 
    in ANY cluster at higher pruning percentages.
    
    Args:
        base_dir: Base directory path
        model: Model name
        exp: Experiment name
        method: Method name
        clusters: List of cluster names
    
    Returns:
        Dictionary containing loss analysis across all clusters
    """

    # Define pruning percentages
    pruning_percentages = ['0.0%Pruned', '25.0%Pruned', '43.75%Pruned', '57.812%Pruned', '68.359%Pruned', '76.27%Pruned']
    
    print(f"\n{'='*60}")
    print(f"Analyzing concept loss ACROSS ALL CLUSTERS for {model}/{exp}/{method}")
    print(f"{'='*60}")
    
    results = {
        'concepts_by_pruning': {},
        'loss_from_baseline': {},
        'sequential_loss': {},
        'foundational':set()
    }
    
    # Load concepts for each pruning percentage (across all clusters)
    baseline_concepts = None
    prev_concepts = None
    
    for pruning_pct in pruning_percentages:
        # Get all concepts across all clusters for this pruning percentage
        all_concepts = get_all_concepts_across_clusters(base_dir, model, exp, method, 
                                                        pruning_pct, clusters,filename)
        
        results['concepts_by_pruning'][pruning_pct] = {
            'all_concepts': all_concepts,
            'num_concepts': len(all_concepts)
        }
        
        # Set baseline (0.0% pruned)
        if baseline_concepts is None:
            baseline_concepts = all_concepts
            print(f"\n{pruning_pct}: {len(all_concepts)} concepts (BASELINE)")
            print(f"  Concepts appear across any of the clusters: {clusters}")
            results['foundational'] = baseline_concepts
        else:
            # Calculate loss from baseline
            lost_from_baseline = baseline_concepts - all_concepts
            retained_from_baseline = baseline_concepts & all_concepts
            results['foundational'] = results['foundational'].intersection(all_concepts)
            
            results['loss_from_baseline'][pruning_pct] = {
                'lost_concepts': lost_from_baseline,
                'num_lost': len(lost_from_baseline),
                'retained_concepts': retained_from_baseline,
                'num_retained': len(retained_from_baseline),
                'loss_percentage': (len(lost_from_baseline) / len(baseline_concepts) * 100) if baseline_concepts else 0,
                'num_foundational': len(lottery_ticket_foundational[0].intersection(all_concepts))/len(lottery_ticket_foundational[0]),
            }
            
            print(f"\n{pruning_pct}: {len(all_concepts)} concepts")
            print(f"  Lost from baseline (not in ANY cluster): {len(lost_from_baseline)} ({results['loss_from_baseline'][pruning_pct]['loss_percentage']:.2f}%)")
            print(f"  Retained from baseline (in at least one cluster): {len(retained_from_baseline)}")
            
            # Calculate sequential loss (compared to previous pruning level)
            if prev_concepts is not None:
                lost_sequential = prev_concepts - all_concepts
                
                results['sequential_loss'][pruning_pct] = {
                    'lost_concepts': lost_sequential,
                    'num_lost': len(lost_sequential),
                    'loss_percentage': (len(lost_sequential) / len(prev_concepts) * 100) if prev_concepts else 0
                }
                
                print(f"  Lost since previous pruning level: {len(lost_sequential)} ({results['sequential_loss'][pruning_pct]['loss_percentage']:.2f}%)")
        
        prev_concepts = all_concepts
    
    return results

def generate_loss_summary(loss_results, save_path=None):
    """
    Generate a summary table of concept loss across pruning percentages.
    
    Args:
        loss_results: Results from analyze_concept_loss_across_clusters
        save_path: Optional path to save CSV summary
    """
    summary_data = []
    print(len(loss_results['foundational']))
    for pruning_pct in loss_results['concepts_by_pruning'].keys():
        row = {
            'Pruning_Percentage': pruning_pct,
            'Total_Concepts': loss_results['concepts_by_pruning'][pruning_pct]['num_concepts']
        }
        
        if pruning_pct in loss_results['loss_from_baseline']:
            row['Lost_from_Baseline'] = loss_results['loss_from_baseline'][pruning_pct]['num_lost']
            row['Lost_from_Baseline_Pct'] = loss_results['loss_from_baseline'][pruning_pct]['loss_percentage']
            row['Retained_from_Baseline'] = loss_results['loss_from_baseline'][pruning_pct]['num_retained']
            row['Foundational'] = loss_results['loss_from_baseline'][pruning_pct]['num_foundational']
        else:
            row['Lost_from_Baseline'] = 0
            row['Lost_from_Baseline_Pct'] = 0
            row['Retained_from_Baseline'] = row['Total_Concepts']
        
        if pruning_pct in loss_results['sequential_loss']:
            row['Lost_Sequential'] = loss_results['sequential_loss'][pruning_pct]['num_lost']
            row['Lost_Sequential_Pct'] = loss_results['sequential_loss'][pruning_pct]['loss_percentage']
        else:
            row['Lost_Sequential'] = 0
            row['Lost_Sequential_Pct'] = 0
        
        summary_data.append(row)
    
    summary_df = pd.DataFrame(summary_data)
    
    if save_path:
        summary_df.to_csv(save_path, index=False)
        print(f"\nSummary saved to {save_path}")
    
    print("\n" + "="*80)
    print("SUMMARY TABLE")
    print("="*80)
    print(summary_df.to_string(index=False))
    
    return summary_df

def save_lost_concepts_details(loss_results, save_dir='lost_concepts_details'):
    """
    Save detailed lists of lost concepts to files.
    
    Args:
        loss_results: Results from analyze_concept_loss_across_clusters
        save_dir: Directory to save the detailed files
    """
    os.makedirs(save_dir, exist_ok=True)
    lost={}
    max_len=0
    for pruning_pct in loss_results['loss_from_baseline'].keys():
        lost_concepts = loss_results['loss_from_baseline'][pruning_pct]['lost_concepts']
        lost[pruning_pct] = sorted(list(lost_concepts))
        max_len = max(max_len, len(lost[pruning_pct]))
    
    for pct in lost:
        dif = max_len-len(lost[pct]) 
        if dif > 0:
            for _ in range(dif):
                lost[pct].append('')
    pd.DataFrame(lost).to_csv(f"lost_cps.csv")
        
       
        
        #print(f"Saved lost concepts to {filename}")

# Example usage:

# Set your paths

base_dir = "/workspace/CCE_NLI"
model = "LLAMA"
exp = "exp"
method = "wanda"
filename='Run0.25_2'

# Analyze concept loss across all clusters
loss_results = analyze_concept_loss_across_clusters(base_dir, model, exp, method,filename=filename)

# Generate summary table
summary_df = generate_loss_summary(loss_results, save_path='concept_loss_summary.csv')

# Save detailed lists of lost concepts
save_lost_concepts_details(loss_results, save_dir='lost_concepts_details')

# Access specific information
print("\n" + "="*80)
print("EXAMPLE: Concepts lost at 25.0%Pruned")
print("="*80)
lost_at_25 = loss_results['loss_from_baseline']['25.0%Pruned']['lost_concepts']
print(f"Total lost: {len(lost_at_25)}")
print(f"First 10 examples: {list(sorted(lost_at_25))[:10]}")


Analyzing concept loss ACROSS ALL CLUSTERS for LLAMA/exp/wanda

0.0%Pruned: 431 concepts (BASELINE)
  Concepts appear across any of the clusters: ['Cluster1', 'Cluster2', 'Cluster3']

25.0%Pruned: 435 concepts
  Lost from baseline (not in ANY cluster): 67 (15.55%)
  Retained from baseline (in at least one cluster): 364
  Lost since previous pruning level: 67 (15.55%)

43.75%Pruned: 410 concepts
  Lost from baseline (not in ANY cluster): 110 (25.52%)
  Retained from baseline (in at least one cluster): 321
  Lost since previous pruning level: 102 (23.45%)

57.812%Pruned: 335 concepts
  Lost from baseline (not in ANY cluster): 152 (35.27%)
  Retained from baseline (in at least one cluster): 279
  Lost since previous pruning level: 132 (32.20%)

68.359%Pruned: 258 concepts
  Lost from baseline (not in ANY cluster): 200 (46.40%)
  Retained from baseline (in at least one cluster): 231
  Lost since previous pruning level: 114 (34.03%)

76.27%Pruned: 191 concepts
  Lost from baseline (not in 

In [102]:
import pandas as pd
lost_concepts= pd.read_csv("/workspace/CCE_NLI/Experiments/lost_concepts_details/lost_cps_llama_lth.csv")
all_lost=set()
all_cps=set()
for i,col in enumerate(lost_concepts.columns):
    if 'Unnamed' in col: continue
    cps = get_concepts(lost_concepts[col])
    if i == 1:
        all_lost=cps
        all_cps=cps
    else:
        all_lost = all_lost.intersection(cps)
        all_cps=all_cps.union(cps)
len(all_lost), len(all_cps)

(33, 173)

In [ ]:
LLAMA wanda Run0.25_2        % of Foundational concpts expld (foundational meaning concepts that were preserved ebd to end in lth pruning)
        0.0%Pruned            NaN
       25.0%Pruned            0.980695
      43.75%Pruned            0.953668
     57.812%Pruned            0.884170
     68.359%Pruned            0.783784
      76.27%Pruned            0.633205

In [95]:
def get_concepts(col):
    cps=set()
    for row in col:
        cps.add(row)
    return cps

In [119]:
lottery_ticket_foundational[0]

{'hyp:tag:.',
 'hyp:tag:cc',
 'hyp:tag:cd',
 'hyp:tag:dt',
 'hyp:tag:ex',
 'hyp:tag:in',
 'hyp:tag:jj',
 'hyp:tag:nn',
 'hyp:tag:nnp',
 'hyp:tag:nns',
 'hyp:tag:prp',
 'hyp:tag:prp$',
 'hyp:tag:rb',
 'hyp:tag:vb',
 'hyp:tag:vbd',
 'hyp:tag:vbg',
 'hyp:tag:vbn',
 'hyp:tag:vbp',
 'hyp:tag:vbz',
 'hyp:tok:after',
 'hyp:tok:alone',
 'hyp:tok:and',
 'hyp:tok:are',
 'hyp:tok:asleep',
 'hyp:tok:at',
 'hyp:tok:baseball',
 'hyp:tok:beach',
 'hyp:tok:because',
 'hyp:tok:bed',
 'hyp:tok:bike',
 'hyp:tok:black',
 'hyp:tok:blue',
 'hyp:tok:boy',
 'hyp:tok:bus',
 'hyp:tok:car',
 'hyp:tok:cat',
 'hyp:tok:chasing',
 'hyp:tok:child',
 'hyp:tok:children',
 'hyp:tok:competition',
 'hyp:tok:cooking',
 'hyp:tok:dancing',
 'hyp:tok:dog',
 'hyp:tok:dogs',
 'hyp:tok:driving',
 'hyp:tok:eating',
 'hyp:tok:enjoying',
 'hyp:tok:first',
 'hyp:tok:for',
 'hyp:tok:friends',
 'hyp:tok:game',
 'hyp:tok:girl',
 'hyp:tok:girls',
 'hyp:tok:guy',
 'hyp:tok:has',
 'hyp:tok:her',
 'hyp:tok:his',
 'hyp:tok:home',
 'hyp:tok:

In [5]:
import os
import pandas as pd
import re
from collections import defaultdict
from itertools import combinations as iter_combinations
import json

def normalize_formula(formula):
    """
    Normalize a formula by:
    1. Removing spaces
    2. Sorting operands within AND/OR groups
    3. Creating a canonical representation
    """
    # Remove spaces
    formula = formula.replace(" ", "")
    
    def sort_expression(expr, operator):
        """Sort operands in an expression by the given operator."""
        # Split by operator (not inside parentheses)
        parts = []
        current = ""
        depth = 0
        
        i = 0
        while i < len(expr):
            if expr[i] == '(':
                depth += 1
                current += expr[i]
            elif expr[i] == ')':
                depth -= 1
                current += expr[i]
            elif depth == 0 and expr[i:i+len(operator)] == operator:
                if current:
                    parts.append(current)
                    current = ""
                i += len(operator) - 1
            else:
                current += expr[i]
            i += 1
        
        if current:
            parts.append(current)
        
        # Sort and rejoin
        if len(parts) > 1:
            parts = sorted(parts)
            return operator.join(parts)
        return expr
    
    # Recursively normalize
    prev = ""
    max_iterations = 10
    iteration = 0
    
    while prev != formula and iteration < max_iterations:
        prev = formula
        iteration += 1
        
        # Process innermost parentheses first
        def normalize_group(match):
            content = match.group(1)
            # Sort by AND first, then OR
            if 'AND' in content:
                content = sort_expression(content, 'AND')
            if 'OR' in content:
                content = sort_expression(content, 'OR')
            return f"({content})"
        
        formula = re.sub(r'\(([^()]+)\)', normalize_group, formula)
    
    return formula


def get_formula_combinations(formula):
    """
    Extract all sub-combinations from a formula, excluding NOT clauses.
    Returns a set of normalized formulas.
    """
    combinations_set = set()
    
    # Remove NOT clauses
    formula_without_not = re.sub(r'\s*(AND|OR)\s*\(NOT[^)]+\)', '', formula)
    formula_without_not = re.sub(r'\(NOT[^)]+\)\s*(AND|OR)\s*', '', formula_without_not)
    formula_without_not = re.sub(r'AND\s*NOT\s+[^\s)]+', '', formula_without_not)
    formula_without_not = re.sub(r'NOT\s+[^\s)]+\s*AND', '', formula_without_not)
    
    # Extract all individual concepts (excluding NOT)
    concepts = []
    pattern = r'(?:hyp|pre|oth):[^\s)()AND|OR]+'
    
    for match in re.finditer(pattern, formula_without_not):
        concept = match.group(0)
        if 'NOT' not in concept and concept not in concepts:
            concepts.append(concept)
    
    if not concepts:
        return combinations_set
    
    # Generate all combinations of concepts
    for r in range(2, len(concepts) + 1):  # Start from 2 to get meaningful combinations
        for combo in iter_combinations(concepts, r):
            # Try both AND and OR
            for operator in [' AND ', ' OR ']:
                combo_formula = f"({operator.join(combo)})"
                normalized = normalize_formula(combo_formula)
                combinations_set.add(normalized)
    
    # Also extract existing complete sub-formulas from the original
    def extract_parenthesized_groups(text):
        """Extract all parenthesized groups that contain operators."""
        results = set()
        stack = []
        
        for i, char in enumerate(text):
            if char == '(':
                stack.append(i)
            elif char == ')' and stack:
                start = stack.pop()
                group = text[start+1:i]
                # Only include if it has AND/OR and no NOT
                if ('AND' in group or 'OR' in group) and 'NOT' not in group:
                    normalized = normalize_formula(f"({group})")
                    results.add(normalized)
        
        return results
    
    combinations_set.update(extract_parenthesized_groups(formula_without_not))
    
    # Remove single concepts (we only want combinations)
    combinations_set = {c for c in combinations_set if 'AND' in c or 'OR' in c}
    
    return combinations_set


def load_formulas_from_csv(filepath):
    """
    Load formulas from a CSV file and extract all combinations.
    """
    if not os.path.exists(filepath):
        return set()
    
    try:
        df = pd.read_csv(filepath)
        if 'best_name' not in df.columns:
            print(f"Warning: 'best_name' column not found in {filepath}")
            return set()
        
        all_combinations = set()
        
        for _, row in df.iterrows():
            formula = str(row['best_name'])
            combinations = get_formula_combinations(formula)
            all_combinations.update(combinations)
        
        return all_combinations
    
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        return set()


def get_combinations_for_method(base_dir, model_name, exp_name, file, method,
                                clusters=['Cluster1', 'Cluster2', 'Cluster3']):
    """
    Get combinations for each pruning percentage for a given method.
    
    Returns:
        dict: {pruning_pct: set of combinations}
    """
    method_dir = os.path.join(base_dir, model_name, 'exp', method, file, 'Expls')
    
    if not os.path.exists(method_dir):
        print(f"Method directory not found: {method_dir}")
        return {}
    
    # Get all pruning percentage directories
    pruning_dirs = []
    for item in os.listdir(method_dir):
        item_path = os.path.join(method_dir, item)
        if os.path.isdir(item_path) and 'pruned' in item.lower():
            pruning_dirs.append(item)
    
    pruning_dirs = sorted(pruning_dirs)
    
    print(f"\nLoading combinations for {method}")
    print(f"Found {len(pruning_dirs)} pruning percentages: {pruning_dirs}")
    
    # Collect combinations for each pruning percentage
    pruning_combinations = {}
    
    for pruning_pct in pruning_dirs:
        all_combos = set()
        
        # Load from all clusters
        for cluster in clusters:
            csv_path = os.path.join(method_dir, pruning_pct, f'{cluster}IOUS1024N.csv')
            combos = load_formulas_from_csv(csv_path)
            all_combos.update(combos)
        
        pruning_combinations[pruning_pct] = all_combos
        print(f"  {pruning_pct}: {len(all_combos)} combinations")
    
    return pruning_combinations


def analyze_lottery_preservation_in_wanda(base_dir, model_name, exp_name,file,
                                          clusters=['Cluster1', 'Cluster2', 'Cluster3']):
    """
    1. Find combinations preserved across ALL lottery_ticket pruning percentages
    2. Check how many of those are present in each wanda pruning percentage
    
    Returns:
        DataFrame with analysis results
    """
    print(f"\n{'='*80}")
    print(f"Analyzing {model_name}/{exp_name}")
    print(f"{'='*80}")
    
    # Step 1: Get lottery_ticket combinations and find preserved ones
    print("\nStep 1: Analyzing lottery_ticket method")
    print("-" * 80)
    lottery_combos = get_combinations_for_method(
        base_dir, model_name, exp_name,file, 'lottery_ticket', clusters
    )
    
    if not lottery_combos:
        print("ERROR: No lottery_ticket data found!")
        return pd.DataFrame()
    
    # Find combinations preserved across ALL lottery pruning percentages
    lottery_preserved = set.intersection(*lottery_combos.values())
    
    print(f"\n{'='*60}")
    print(f"Lottery Ticket - Preserved across ALL pruning percentages:")
    print(f"  Total preserved: {len(lottery_preserved)}")
    print(f"{'='*60}")
    
    # Step 2: Get wanda combinations
    print("\nStep 2: Analyzing wanda method")
    print("-" * 80)
    wanda_combos = get_combinations_for_method(
        base_dir, model_name, exp_name, 'wanda', clusters
    )
    
    if not wanda_combos:
        print("ERROR: No wanda data found!")
        return pd.DataFrame()
    
    # Step 3: Check how many lottery-preserved combos are in each wanda pruning %
    print(f"\nStep 3: Checking lottery-preserved combinations in wanda")
    print("-" * 80)
    
    results = []
    
    for pruning_pct in sorted(wanda_combos.keys()):
        wanda_set = wanda_combos[pruning_pct]
        
        # How many lottery-preserved combos are in this wanda pruning %
        present_in_wanda = lottery_preserved & wanda_set
        
        num_present = len(present_in_wanda)
        num_lottery_preserved = len(lottery_preserved)
        preservation_rate = (num_present / num_lottery_preserved * 100) if num_lottery_preserved > 0 else 0
        
        results.append({
            'Model': model_name,
            'Experiment': exp_name,
            'Pruning_Percentage': pruning_pct,
            'Lottery_Preserved_Total': num_lottery_preserved,
            'Present_in_Wanda': num_present,
            'Missing_in_Wanda': num_lottery_preserved - num_present,
            'Preservation_Rate_%': preservation_rate
        })
        
        print(f"  {pruning_pct}:")
        print(f"    Present: {num_present}/{num_lottery_preserved} ({preservation_rate:.2f}%)")
        print(f"    Missing: {num_lottery_preserved - num_present}")
    
    return pd.DataFrame(results), lottery_preserved


def save_preserved_combinations(lottery_preserved, output_file='lottery_preserved_combinations.json'):
    """Save the lottery-preserved combinations to a file."""
    preserved_list = sorted(list(lottery_preserved))
    
    with open(output_file, 'w') as f:
        json.dump(preserved_list, f, indent=2)
    
    print(f"\nLottery-preserved combinations saved to '{output_file}'")
    print(f"Total combinations saved: {len(preserved_list)}")
    
    # Show first 10 examples
    print("\nFirst 10 examples of preserved combinations:")
    for i, combo in enumerate(preserved_list[:10], 1):
        print(f"  {i}. {combo}")


# Example usage
if __name__ == "__main__":
    # Set your paths
    base_dir = "/workspace/CCE_NLI/"
    model_name = "BOWMAN"
    exp_name = "exp"
    file = 'Run0.25_3'
    
    # Run analysis
    results_df, lottery_preserved = analyze_lottery_preservation_in_wanda(
        base_dir, 
        model_name, 
        exp_name,
        file,
        clusters=['Cluster1', 'Cluster2', 'Cluster3']
    )
    
    # Display results
    print("\n" + "="*80)
    print("ANALYSIS RESULTS")
    print("="*80)
    print(results_df.to_string(index=False))
    
    # Save results
    results_df.to_csv('lottery_preservation_in_wanda.csv', index=False)
    print(f"\nResults saved to 'lottery_preservation_in_wanda.csv'")
    
    # Save preserved combinations
    save_preserved_combinations(lottery_preserved)


Analyzing BOWMAN/exp

Step 1: Analyzing lottery_ticket method
--------------------------------------------------------------------------------

Loading combinations for lottery_ticket
Found 7 pruning percentages: ['0.0%Pruned', '25.0%Pruned', '43.75%Pruned', '57.812%Pruned', '68.359%Pruned', '76.27%Pruned', '82.202%Pruned']
  0.0%Pruned: 660 combinations
  25.0%Pruned: 670 combinations
  43.75%Pruned: 765 combinations
  57.812%Pruned: 821 combinations
  68.359%Pruned: 1117 combinations
  76.27%Pruned: 1044 combinations
  82.202%Pruned: 1195 combinations

Lottery Ticket - Preserved across ALL pruning percentages:
  Total preserved: 32

Step 2: Analyzing wanda method
--------------------------------------------------------------------------------


TypeError: join() argument must be str, bytes, or os.PathLike object, not 'list'

In [31]:
global_loterry_preserved=[set()]

In [32]:
import os
import pandas as pd
import re
from collections import defaultdict
import json

def get_indiv_concepts(formula) -> list:
    concepts = []
    concps = re.findall(r'(?<!\bNOT\s)(?:\b(?:hyp|pre|oth):[^\s)]+)', formula)
    for c in concps:
        try:
            end_idx = c.index(')')
        except:
            end_idx = len(c)
        concepts.append(c[:end_idx])
    return concepts

def load_concepts_from_csv(filepath):
    """
    Load all concepts from a CSV file.
    
    Args:
        filepath: Path to CSV file
    
    Returns:
        set: Set of all unique concepts in the file
    """
    if not os.path.exists(filepath):
        print(f"Warning: File not found - {filepath}")
        return set()
    
    try:
        df = pd.read_csv(filepath)
        
        if 'best_name' not in df.columns:
            print(f"Warning: 'best_name' column not found in {filepath}")
            return set()
        
        all_concepts = set()
        
        for _, row in df.iterrows():
            formula = str(row['best_name'])
            concepts = get_indiv_concepts(formula)
            all_concepts.update(concepts)
        
        return all_concepts
    
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        return set()


def get_concepts_for_method(base_dir, model_name, exp_name, file, method,
                           clusters=['Cluster1', 'Cluster2', 'Cluster3']):
    """
    Get concepts for each pruning percentage for a given method.
    
    Args:
        base_dir: Base directory path
        model_name: Model name
        exp_name: Experiment name
        method: Method name (e.g., 'lottery_ticket', 'wanda')
        clusters: List of cluster names
    
    Returns:
        dict: {pruning_pct: set of concepts}
    """
    method_dir = os.path.join(base_dir, model_name, exp_name, method, file, 'Expls')
    
    if not os.path.exists(method_dir):
        print(f"Method directory not found: {method_dir}")
        return {}
    
    # Get all pruning percentage directories
    pruning_dirs = []
    for item in os.listdir(method_dir):
        item_path = os.path.join(method_dir, item)
        if os.path.isdir(item_path) and 'pruned' in item.lower():
            pruning_dirs.append(item)
    
    pruning_dirs = sorted(pruning_dirs)
    
    print(f"\n{'='*80}")
    print(f"Loading concepts for {model_name}/{exp_name}/{method}")
    print(f"Found {len(pruning_dirs)} pruning percentages: {pruning_dirs}")
    print(f"{'='*80}")
    
    # Collect concepts for each pruning percentage
    pruning_concepts = {}
    
    for pruning_pct in pruning_dirs:
        all_concepts = set()
        
        # Load from all clusters (don't distinguish between clusters)
        for cluster in clusters:
            csv_path = os.path.join(method_dir, pruning_pct, f'{cluster}IOUS1024N.csv')
            concepts = load_concepts_from_csv(csv_path)
            all_concepts.update(concepts)
            
            if concepts:
                print(f"  {pruning_pct}/{cluster}: {len(concepts)} unique concepts")
        
        pruning_concepts[pruning_pct] = all_concepts
        print(f"  {pruning_pct} TOTAL: {len(all_concepts)} unique concepts (across all clusters)")
    
    return pruning_concepts


def analyze_preserved_concepts(base_dir, model_name, exp_name, file, methods=['lottery_ticket', 'wanda'],
                               clusters=['Cluster1', 'Cluster2', 'Cluster3']):
    """
    Analyze which concepts are preserved across all pruning percentages for each method.
    
    Args:
        base_dir: Base directory path
        model_name: Model name
        exp_name: Experiment name
        methods: List of methods
        clusters: List of cluster names
    
    Returns:
        DataFrame with results
    """
    results = []
    
    for method in methods:
        # Get concepts for each pruning percentage
        pruning_concepts = get_concepts_for_method(
            base_dir, model_name, exp_name, file, method, clusters
        )
        
        if not pruning_concepts:
            print(f"No data found for method: {method}")
            continue
        
        # Find concepts preserved across ALL pruning percentages
        preserved_concepts = set.intersection(*pruning_concepts.values())
        
        print(f"\n{'='*60}")
        print(f"{method.upper()}: Concepts preserved across ALL pruning percentages")
        print(f"  Total preserved: {len(preserved_concepts)}")
        print(f"{'='*60}")
        
        # Create result for each pruning percentage
        for pruning_pct in sorted(pruning_concepts.keys()):
            total_concepts = len(pruning_concepts[pruning_pct])
            preserved_count = len(preserved_concepts)
            preservation_rate = (preserved_count / total_concepts * 100) if total_concepts > 0 else 0
            
            results.append({
                'Model': model_name,
                'Experiment': exp_name,
                'Method': method,
                'Pruning_Percentage': pruning_pct,
                'Total_Concepts': total_concepts,
                'Preserved_Concepts': preserved_count,
                'Preservation_Rate_%': preservation_rate
            })
            
            print(f"  {pruning_pct}:")
            print(f"    Total concepts: {total_concepts}")
            print(f"    Preserved: {preserved_count} ({preservation_rate:.2f}%)")
    
    return pd.DataFrame(results)


def get_preserved_concepts_details(base_dir, model_name, exp_name, file, method,
                                   clusters=['Cluster1', 'Cluster2', 'Cluster3']):
    """
    Get the actual list of preserved concepts for a method.
    
    Returns:
        list: Sorted list of preserved concepts
    """
    pruning_concepts = get_concepts_for_method(
        base_dir, model_name, exp_name, file, method, clusters
    )
    
    if not pruning_concepts:
        return []
    
    # Find preserved concepts
    preserved = set.intersection(*pruning_concepts.values())
    
    return sorted(preserved)


def save_preserved_concepts(preserved_concepts, method, output_file=None):
    """
    Save preserved concepts to a JSON file.
    
    Args:
        preserved_concepts: List of preserved concepts
        method: Method name
        output_file: Optional output filename
    """
    if output_file is None:
        output_file = f'preserved_concepts_{method}.json'
    
    with open(output_file, 'w') as f:
        json.dump(preserved_concepts, f, indent=2)
    
    print(f"\nPreserved concepts for {method} saved to '{output_file}'")
    print(f"Total concepts saved: {len(preserved_concepts)}")
    
    # Show first 20 examples
    print(f"\nFirst 20 preserved concepts:")
    for i, concept in enumerate(preserved_concepts[:20], 1):
        print(f"  {i:2d}. {concept}")
    
    if len(preserved_concepts) > 20:
        print(f"  ... and {len(preserved_concepts) - 20} more")


def compare_methods_preservation(base_dir, model_name, exp_name,
                                clusters=['Cluster1', 'Cluster2', 'Cluster3']):
    """
    Compare concept preservation between lottery_ticket and wanda methods.
    """
    print(f"\n{'='*80}")
    print(f"COMPARING METHODS: lottery_ticket vs wanda")
    print(f"{'='*80}")
    
    # Get preserved concepts for each method
    lottery_preserved = set(get_preserved_concepts_details(
        base_dir, model_name, exp_name, 'Run0.25_3', 'lottery_ticket', clusters
    ))
    
    
    wanda_preserved = set(get_preserved_concepts_details(
        base_dir, model_name, exp_name, 'wanda', clusters
    ))
    
    # Find overlaps and differences
    both_preserved = lottery_preserved & wanda_preserved
    only_lottery = lottery_preserved - wanda_preserved
    only_wanda = wanda_preserved - lottery_preserved
    
    print(f"\nPreserved in both methods: {len(both_preserved)}")
    print(f"Preserved only in lottery_ticket: {len(only_lottery)}")
    print(f"Preserved only in wanda: {len(only_wanda)}")
    
    comparison = {
        'lottery_ticket_preserved': len(lottery_preserved),
        'wanda_preserved': len(wanda_preserved),
        'both_preserved': len(both_preserved),
        'only_lottery': len(only_lottery),
        'only_wanda': len(only_wanda),
        'overlap_rate_%': (len(both_preserved) / len(lottery_preserved) * 100) if lottery_preserved else 0
    }
    
    return comparison, both_preserved, only_lottery, only_wanda


# Example usage
if __name__ == "__main__":
    # Set your paths
    base_dir = "/workspace/CCE_NLI"
    model_name = "BOWMAN"
    exp_name = "exp"
    file='Run0.25_4'
    
    # Analyze preservation for each method
    results_df = analyze_preserved_concepts(
        base_dir, 
        model_name, 
        exp_name,
        file,
        methods=['lottery_ticket', 'wanda'],
        clusters=['Cluster1', 'Cluster2', 'Cluster3']
    )
    
    # Display results
    print("\n" + "="*80)
    print("PRESERVATION ANALYSIS RESULTS")
    print("="*80)
    print(results_df.to_string(index=False))
    
    # Save results
    results_df.to_csv('concept_preservation_analysis.csv', index=False)
    print(f"\nResults saved to 'concept_preservation_analysis.csv'")
    
    # Get and save preserved concepts for each method
    print("\n" + "="*80)
    print("PRESERVED CONCEPTS DETAILS")
    print("="*80)
    
    lottery_preserved = get_preserved_concepts_details(
        base_dir, model_name, exp_name, file, 'lottery_ticket'
    )
    global_loterry_preserved[0] = lottery_preserved
    save_preserved_concepts(lottery_preserved, 'lottery_ticket')
    
    wanda_preserved = get_preserved_concepts_details(
        base_dir, model_name, exp_name, 'wanda'
    )
    save_preserved_concepts(wanda_preserved, 'wanda')
    
    # Compare methods
    comparison, both, only_lottery, only_wanda = compare_methods_preservation(
        base_dir, model_name, exp_name
    )
    
    print("\n" + "="*80)
    print("METHOD COMPARISON")
    print("="*80)
    for key, value in comparison.items():
        print(f"{key}: {value}")
    
    # Save comparison results
    with open('method_comparison.json', 'w') as f:
        json.dump({
            'comparison': comparison,
            'both_preserved': sorted(both),
            'only_lottery': sorted(only_lottery),
            'only_wanda': sorted(only_wanda)
        }, f, indent=2)
    
    print("\nComparison saved to 'method_comparison.json'")


Loading concepts for BOWMAN/exp/lottery_ticket
Found 7 pruning percentages: ['0.0%Pruned', '25.0%Pruned', '43.75%Pruned', '57.812%Pruned', '68.359%Pruned', '76.27%Pruned', '82.202%Pruned']
  0.0%Pruned/Cluster1: 27 unique concepts
  0.0%Pruned/Cluster2: 30 unique concepts
  0.0%Pruned/Cluster3: 8 unique concepts
  0.0%Pruned TOTAL: 49 unique concepts (across all clusters)
  25.0%Pruned/Cluster1: 51 unique concepts
  25.0%Pruned/Cluster2: 36 unique concepts
  25.0%Pruned/Cluster3: 15 unique concepts
  25.0%Pruned TOTAL: 68 unique concepts (across all clusters)
  43.75%Pruned/Cluster1: 50 unique concepts
  43.75%Pruned/Cluster2: 40 unique concepts
  43.75%Pruned/Cluster3: 5 unique concepts
  43.75%Pruned TOTAL: 70 unique concepts (across all clusters)
  57.812%Pruned/Cluster1: 44 unique concepts
  57.812%Pruned/Cluster2: 41 unique concepts
  57.812%Pruned/Cluster3: 8 unique concepts
  57.812%Pruned TOTAL: 68 unique concepts (across all clusters)
  68.359%Pruned/Cluster1: 50 unique conce

TypeError: get_preserved_concepts_details() missing 1 required positional argument: 'method'

In [33]:
global_loterry_preserved[0]

['hyp:tag:.',
 'hyp:tag:dt',
 'hyp:tag:ex',
 'hyp:tag:in',
 'hyp:tag:nn',
 'hyp:tag:prp$',
 'hyp:tok:after',
 'hyp:tok:eating',
 'hyp:tok:for',
 'hyp:tok:friends',
 'hyp:tok:in',
 'hyp:tok:near',
 'hyp:tok:nobody',
 'hyp:tok:outdoors',
 'hyp:tok:outside',
 'hyp:tok:people',
 'hyp:tok:running',
 'hyp:tok:sitting',
 'hyp:tok:sleeping',
 'hyp:tok:swimming',
 'hyp:tok:tall',
 'hyp:tok:to',
 'oth:overlap:overlap25',
 'oth:overlap:overlap50',
 'oth:overlap:overlap75',
 'pre:tag:.',
 'pre:tag:nn']

In [44]:
import os
import pandas as pd
import re

def extract_concepts(formula):
    """
    Extract all individual concepts from a formula, excluding NOT clauses.
    
    Args:
        formula: String like "((((hyp:tok:for OR hyp:tok:to) OR hyp:tok:tall) OR hyp:tag:prp) AND (NOT oth:overlap:overlap75))"
    
    Returns:
        set: Set of individual concepts
    """
    concepts = set()
    
    # Remove all NOT clauses first
    formula_without_not = re.sub(r'\(NOT[^)]+\)', '', formula)
    formula_without_not = re.sub(r'NOT\s+[^\s)]+', '', formula_without_not)
    
    # Extract concepts from cleaned formula
    pattern = r'(?:hyp|pre|oth):[^\s)()AND|OR]+'
    matches = re.findall(pattern, formula_without_not)
    
    for match in matches:
        concepts.add(match)
    
    return concepts


def find_neurons_with_concepts(folder_path, target_concepts, 
                               clusters=['Cluster1', 'Cluster2', 'Cluster3']):
    """
    Find all neurons (units) that have at least one concept from the target list.
    
    Args:
        folder_path: Path to the folder containing ClusterXIOU1024N.csv files
        target_concepts: List or set of concepts to search for
        clusters: List of cluster names to search
    
    Returns:
        list: List of dictionaries with cluster, unit, and matching concepts
    """
    target_concepts = set(target_concepts)  # Convert to set for faster lookup
    results = []
    
    print(f"\n{'='*80}")
    print(f"Searching for neurons with target concepts in: {folder_path}")
    print(f"Target concepts: {sorted(target_concepts)}")
    print(f"{'='*80}\n")
    
    for cluster in clusters:
        csv_path = os.path.join(folder_path, f'{cluster}IOUS1024N.csv')
        
        if not os.path.exists(csv_path):
            print(f"Warning: File not found - {csv_path}")
            continue
        
        try:
            df = pd.read_csv(csv_path)
            
            if 'unit' not in df.columns or 'best_name' not in df.columns:
                print(f"Warning: Required columns not found in {csv_path}")
                continue
            
            print(f"Searching in {cluster}...")
            cluster_matches = 0
            droppable_units=[]
            for _, row in df.iterrows():
                unit = row['unit']
                formula = str(row['best_name'])
                
                # Extract concepts from this unit's formula
                unit_concepts = extract_concepts(formula)
                
                # Check if any target concepts are in this unit
                matching_concepts = unit_concepts & target_concepts
                
                if matching_concepts:
                    results.append({
                        'cluster': cluster,
                        'unit': unit,
                        'matching_concepts': sorted(matching_concepts),
                        'num_matches': len(matching_concepts),
                        'formula': formula
                    })
                    cluster_matches += 1
                else:
                    droppable_units.append(unit)
            
            print(f"  Found {cluster_matches} neurons with target concepts\n {len(df) - cluster_matches} neurons don't explain foundationals")
            if len(df) - cluster_matches > 0:
                print(f"Droppable units: {droppable_units}")
                
        
        except Exception as e:
            print(f"Error reading {csv_path}: {e}\n")
    
    return results
def pretty_print(results):
    for cluster in ['Cluster1', 'Cluster2', 'Cluster3']:
        print("=="*80)
        print(cluster)
        print("=="*80)
        for res in results:
            if res['cluster'] == cluster:
                
                
                print(res['unit'], res['matching_concepts'], res['formula'] )
        print("\n\n\n")
#with bowman foundational concepts are in eery neuron
# Example usage
if __name__ == "__main__":
    # Specify the exact folder path
    folder_path = '/workspace/CCE_NLI/BOWMAN/exp/lottery_ticket/Run0.25_4/Expls/57.812%Pruned/'
    
    # Specify the concepts you're looking for
    target_concepts = global_loterry_preserved[0]
    # Find neurons
    results=find_neurons_with_concepts(folder_path, target_concepts)
    #pretty_print(results)
   


Searching for neurons with target concepts in: /workspace/CCE_NLI/BOWMAN/exp/lottery_ticket/Run0.25_4/Expls/57.812%Pruned/
Target concepts: ['hyp:tag:.', 'hyp:tag:dt', 'hyp:tag:ex', 'hyp:tag:in', 'hyp:tag:nn', 'hyp:tag:prp$', 'hyp:tok:after', 'hyp:tok:eating', 'hyp:tok:for', 'hyp:tok:friends', 'hyp:tok:in', 'hyp:tok:near', 'hyp:tok:nobody', 'hyp:tok:outdoors', 'hyp:tok:outside', 'hyp:tok:people', 'hyp:tok:running', 'hyp:tok:sitting', 'hyp:tok:sleeping', 'hyp:tok:swimming', 'hyp:tok:tall', 'hyp:tok:to', 'oth:overlap:overlap25', 'oth:overlap:overlap50', 'oth:overlap:overlap75', 'pre:tag:.', 'pre:tag:nn']

Searching in Cluster1...
  Found 31 neurons with target concepts
 0 neurons don't explain foundationals
Searching in Cluster2...
  Found 19 neurons with target concepts
 2 neurons don't explain foundationals
Droppable units: [386, 683]
Searching in Cluster3...
  Found 2 neurons with target concepts
 0 neurons don't explain foundationals
